# **Downstream analysis of pseudobulk results**

This notebook analyzes pseudobulk behavior of genes: Alzheimer's vs control (DS2, 5, 8), Parkinson's vs control (DS1), ALS vs control (DS3), FTLD vs control (DS3), and aged vs young (DS4, 6, 9). 

In [ ]:
# Import important libraries

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
# Load the integrated cross-dataset pseudobulk matrix with semicolon separator
file_path = "Pseudobulk_Results.csv"

# Enforce ';' as the delimiter to handle Excel-exported CSV formatting correctly
df_pb_summary = pd.read_csv(file_path, sep=';', index_col=0)

# Display framework dimensions and verify integrated data structure
print(f"Pseudobulk summary table successfully loaded. Shape: {df_pb_summary.shape}")

In [ ]:
df_pb_summary

In [ ]:
def plot_genes_summary(summary_df, gene_list):
    """
    Plots a summary bar plot for a given list of genes using the integrated matrix.
    Colors follow a strict palette:
    - Enriched/Activated (Mean LFC > 0) = Yellow (#FDFD96)
    - Suppressed/Repressed (Mean LFC < 0) = Purple (#B19CD9)
    """
    # Standard cross-dataset identifiers matching table column prefixes
    data_sources = [
        ('D1', 'DS1'), ('D2', 'DS2'), ('D3_AC', 'DS3 (A)'),
        ('D3_FC', 'DS3 (F)'), ('D4', 'DS4'), ('D5', 'DS5'),
        ('D6', 'DS6'), ('D8', 'DS8'), ('D9', 'DS9')
    ]
    
    # Original cross-dataset color coding definition for individual dot markers
    rich_colors = {
        'DS1': '#4CAF50', 'DS2': '#F4B86C', 'DS3 (A)': '#C96BF5',
        'DS3 (F)': '#6CC3F4', 'DS4': '#7FFFD4', 'DS5': '#FF6161',
        'DS6': '#6DE26A', 'DS8': '#4C9BE8', 'DS9': '#F46CB8',
    }
    
    rows = []
    
    # 1. Structural data harvesting from the semicolon-separated summary dataframe
    for gene in gene_list:
        if gene not in summary_df.index:
            continue
            
        vals = []
        for prefix, label in data_sources:
            lfc_col = f"{prefix}_log2FoldChange"
            
            # Check if dataset columns exist and value is present
            if lfc_col in summary_df.columns:
                lfc_val = summary_df.at[gene, lfc_col]
                if not pd.isna(lfc_val):
                    vals.append((label, lfc_val))
                    
        if vals:
            mean_lfc = np.mean([x[1] for x in vals])
            rows.append({'gene': gene, 'values': vals, 'mean': mean_lfc})
            
    if not rows:
        print("Error: No data found across datasets for the requested gene list.")
        return
        
    # Reverse the order to ensure the first input gene appears at the very TOP
    plot_data = rows[::-1]
    
    # 2. Dynamic canvas geometry calculations based on input row density
    height_per_gene = 0.6  
    fig_width = 14         
    fig_height = len(plot_data) * height_per_gene + 2
    
    plt.close('all')
    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=300)
    plt.subplots_adjust(left=0.25, right=0.8, top=0.95, bottom=0.1)
    
    y_pos = np.arange(len(plot_data))
    
    # 3. Layered multi-graph rendering engine execution
    for idx, item in enumerate(plot_data):
        mv = item['mean']
        
        # Enforce original color code definitions: Yellow for positive, Purple for negative
        bar_color = '#FDFD96' if mv > 0 else '#B19CD9'
        
        # Plot background bar layers
        ax.barh(y_pos[idx], mv, color=bar_color, height=0.8, alpha=1.0,
                edgecolor='black', linewidth=0.5, zorder=2)
        
        # Scatter individual dataset data points with cross-dataset color coding
        for ds_label, val in item['values']:
            jitter = np.random.uniform(-0.25, 0.25)
            ax.scatter(val, y_pos[idx] + jitter, 
                       color=rich_colors.get(ds_label, 'grey'),
                       s=100, edgecolor='black', linewidth=0.8, zorder=5, alpha=1.0)
            
    # 4. Typography and axis decoration styling adjustments
    ax.axvline(0, color='#333333', linewidth=2, zorder=3)
    ax.set_xlabel('log2FoldChange', fontsize=18, fontweight='bold')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels([item['gene'] for item in plot_data],
                       fontsize=16, fontweight='bold', fontstyle='italic')
    ax.tick_params(axis='x', labelsize=14)
    ax.grid(axis='x', linestyle='--', alpha=0.4, zorder=0)
    
    # 5. Restructured cross-dataset legend layout compilation
    legend_handles = [
        Line2D([], [], marker='o', color='w', label=label,
               markerfacecolor=rich_colors.get(label), markersize=10,
               markeredgecolor='black') for _, label in data_sources
    ]
    ax.legend(handles=legend_handles, title='Datasets', title_fontsize=14, fontsize=12,
              bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0, frameon=False)
    
    sns.despine(left=True)
    
    # Save a high-resolution checkpoint PNG
    plt.savefig('my_genes_plot.png', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
plot_genes_summary(
    df_pb_summary,
    ["IL6ST", "CX3CR1", "P2RY12", "ITPR2", "NAV3", "MEF2C", 
    "SRGAP2", "SRGAP2B", "DOCK8", "MEF2A", "FRMD4A", "ARHGAP22", "SORL1", 
    "ANKRD44", "AOAH"])

Homeostatic genes decrease concordantly in aging and disease. 

In [ ]:
plot_genes_summary(
    df_pb_summary,
    ["GPNMB", "IQGAP2", "DPYD", "PTPRG", "MITF", "PPARG",
     "KCNMA1", "LRRK2", "FOXP1", "FKBP5", "NHSL1", "TANC2",  
     "FMN1", "STARD13", "FMNL2", "MYO1E", "SLC11A1", "CPM", "SPP1"])

GPNMB+ signature genes increase in aging and disease.

In [ ]:
plot_genes_summary(
    df_pb_summary,
    ["TPT1", "TMSB10", "TMSB4X", "FTL", "FTH1", "RPS27A", "RPL23", "RPL19", "RPS11", "RPS6", 
    "RPLP1", "RPL13A", "RPL13", "APOE", "C1QB", "HLA-DRA", "HLA-C", "HLA-B", 
    "PLEKHA7", "MECOM", "OOEP", 
    "HSPA1A", "HSPB1", "HSPH1", "DNAJB1", 'HSP90AA1'])

Unexpectedly for us, ribosomal and most of their related genes (including TPT1, TMSB4X, TMSB10, PLEKHA7, MECOM, and OOEP, but notably excluding C1Qs and APOE) decrease in aging and disease. HLAs and HSPs also decline.